In [2]:
# !pip install requests beautifulsoup4 pandas lxml --break-system-packages


In [3]:
from bs4 import BeautifulSoup
import re
import os
import pandas as pd


## Getting Players Information

In [4]:
def find_general_info(soup, url):
    info = {
        'full_name': None, 'birth_date': None, 'born': None,
        'height': None, 'weight': None, 'nationality': None,
        'position': None, 'shoots': None, 'college': None,
        'high_school': None, 'career_length': None, 'link': url,
    }

    meta = soup.find('div', id='meta')
    if meta is None:
        return info

    h1 = meta.find('h1')
    if h1:
        info['full_name'] = h1.text.strip()

    nat_span = meta.find('span', class_='f-i')
    if nat_span:
        info['nationality'] = nat_span.get_text(strip=True)

    for p in meta.find_all('p'):
        # text = p.get_text(strip=True)
        text = re.sub(r'\s+', ' ', p.get_text(strip=True)).strip()
        try:
            if 'Career Length' in text:
                info['career_length'] = text.replace('Career Length:', '').strip()

            elif 'Position' in text:
                m = re.search(r'Position:\s*(.+)', text)
                if m:
                    pos = m.group(1)
                    # strip off a trailing 'Shoots:' fragment if present on the same line
                    pos = re.split(r'Shoots:', pos)[0].strip(' \u25aa')
                    info['position'] = pos.strip()
                m2 = re.search(r'Shoots:\s*(\w+)', text)
                if m2:
                    info['shoots'] = m2.group(1).strip()

            elif 'College' in text:
                info['college'] = text.replace('College:', '').strip()

            elif 'High School' in text:
                info['high_school'] = text.replace('High School:', '').strip()

            elif 'Born' in text:
                m = re.search(r'Born:\s*(.*?)\s*in\b', text)
                if m:
                    info['birth_date'] = m.group(1).strip()
                m2 = re.search(r'in(.*)', text)
                if m2:
                    info['born'] = m2.group(1).strip()

            # e.g. 6-5,205lb(196cm, 92kg)
            elif 'lb' in text or 'kg' in text or 'cm' in text:
                m = re.search(r'\((\d{3})cm', text)
                if m:
                    info['height'] = m.group(1).strip()
                m2 = re.search(r'(\d{2,3})kg', text)
                if m2:
                    info['weight'] = m2.group(1).strip()
        except Exception:
            # don't let one malformed field kill the whole player
            continue

    return info


def find_stats(soup):
    stats = {}
    for div in soup.select(".stats_pullout .p1 > div, .stats_pullout .p2 > div, .stats_pullout .p3 > div"):
        try:
            strong = div.find("strong")
            if strong is None:
                continue
            key = strong.get_text(strip=True)
            ps = div.find_all("p")
            if not ps:
                continue
            value = ps[-1].get_text(strip=True)
            stats[key] = float(value)
        except (ValueError, AttributeError):
            continue
    return stats

In [6]:
# Step 1: List the files and open one
import os

PATH = '../nba_player_html'
data_dir = os.listdir(PATH)
print(f'{len(data_dir)} html have been scraped successfully.')

5416 html have been scraped successfully.


In [7]:
for h in data_dir[:5]:
    print(h)

aubucch01.html
willial06.html
barnhle01.html
morriis01.html
jacksja01.html


In [9]:
results = []
BASE = "https://www.basketball-reference.com"
for html in data_dir:
    p = os.path.join(PATH, html)
    with open(p, 'r', encoding='utf-8') as file:
        content = file.read()
    
    soup = BeautifulSoup(content, 'lxml')
    url = BASE + f'/players/{html[0]}/{html}'
    info = find_general_info(soup, url)
    stats = find_stats(soup)
    info.update(stats)
    results.append(info)


In [10]:
df = pd.DataFrame(results)
df.head()

,full_name,birth_date,born,height,weight,nationality,position,shoots,college,high_school,...,G,PTS,AST,FG%,FT%,WS,TRB,FG3%,eFG%,PER
0,Chet Aubuchon,"May 18,1916","Gary,Indianaus",178,62,us,Guard,Right,Michigan State,"Horace Mann in Gary,Indiana",...,30.0,2.2,0.7,25.3,54.3,1.2,NaN,NaN,NaN,NaN
1,Alondes Williams,"June 19,1999","Milwaukee ,Wisconsinus",193,95,us,Shooting Guard,Right,"Colleges:Triton College,Oklahoma,Wake Forest","Riverside University in Milwaukee,Wisconsin",...,13.0,4.2,1.0,55.6,88.9,0.2,2.1,31.6,63.9,15.1
2,Leo Barnhorst,"May 11,1924","Indianapolis,Indianaus",193,86,us,Small Forward,Right,Notre Dame,"Cathedral in Indianapolis,Indiana",...,344.0,9.4,3.2,36.8,66.5,13.3,5.4,NaN,NaN,14.2
3,Isaiah Morris,"April 2,1969","Richmond,Virginiaus",203,103,us,Small Forward,Right,Arkansas,"Huguenot in Richmond,Virginia",...,25.0,2.2,0.2,45.6,75.0,0.0,0.5,NaN,45.6,10.6
4,Jaren Jackson,"October 27,1967","New Orleans,Louisianaus",193,86,us,Shooting Guard,Right,Georgetown,"Walter Cohen in New Orleans,Louisiana",...,431.0,5.5,1.2,39.1,76.4,11.5,1.8,35.3,46.9,10.1


In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5416 entries, 0 to 5415
Data columns (total 22 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   full_name      5416 non-null   str    
 1   birth_date     5309 non-null   str    
 2   born           5412 non-null   str    
 3   height         5416 non-null   str    
 4   weight         5411 non-null   str    
 5   nationality    5416 non-null   str    
 6   position       5416 non-null   str    
 7   shoots         5416 non-null   str    
 8   college        5015 non-null   str    
 9   high_school    4867 non-null   str    
 10  career_length  4686 non-null   str    
 11  link           5416 non-null   str    
 12  G              5416 non-null   float64
 13  PTS            5416 non-null   float64
 14  AST            5416 non-null   float64
 15  FG%            5381 non-null   float64
 16  FT%            5164 non-null   float64
 17  WS             5415 non-null   float64
 18  TRB            5124

In [13]:
df.isnull().sum()

full_name           0
birth_date        107
born                4
height              0
weight              5
nationality         0
position            0
shoots              0
college           401
high_school       549
career_length     730
link                0
G                   0
PTS                 0
AST                 0
FG%                35
FT%               252
WS                  1
TRB               292
FG3%             1665
eFG%             1157
PER               348
dtype: int64

In [14]:
sample_data_path = os.path.join('../Data/raw', 'players.csv')
df.to_csv(sample_data_path)

In [77]:
df.head()

,full_name,birth_date,born,height,weight,nationality,position,shoots,college,high_school,...,G,PTS,AST,FG%,FT%,WS,TRB,PER,eFG%,FG3%
0,Chet Aubuchon,"May 18,1916","Gary,Indianaus",178,62,us,Guard,Right,Michigan State,"Horace Mann in Gary,Indiana",...,30.0,2.2,0.7,25.3,54.3,1.2,NaN,NaN,NaN,NaN
1,Leo Barnhorst,"May 11,1924","Indianapolis,Indianaus",193,86,us,Small Forward,Right,Notre Dame,"Cathedral in Indianapolis,Indiana",...,344.0,9.4,3.2,36.8,66.5,13.3,5.4,14.2,NaN,NaN
2,Isaiah Morris,"April 2,1969","Richmond,Virginiaus",203,103,us,Small Forward,Right,Arkansas,"Huguenot in Richmond,Virginia",...,25.0,2.2,0.2,45.6,75.0,0.0,0.5,10.6,45.6,NaN
3,Jaren Jackson,"October 27,1967","New Orleans,Louisianaus",193,86,us,Shooting Guard,Right,Georgetown,"Walter Cohen in New Orleans,Louisiana",...,431.0,5.5,1.2,39.1,76.4,11.5,1.8,10.1,46.9,35.3
4,Gui Santos,"June 22,2002","Brasilia,Brazilbr",201,83,br,Power Forward,Right,NaN,NaN,...,147.0,6.4,1.7,49.0,73.7,5.2,3.3,13.5,57.8,34.7
